# QLoRA Fine-Tuning — lectureOS SFT

Fine-tunes `Qwen/Qwen2.5-7B-Instruct` on the synthetic QA pairs generated
from lecture transcripts using 4-bit NF4 quantisation + LoRA adapters.

**Runtime:** GPU (A100/T4 on Colab Pro, or any CUDA device with ≥16 GB VRAM)  
**Estimated time:** ~90 min on A100 for 3 epochs over 5 592 examples

## 0 · Mount Drive & set paths

In [ ]:
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to wherever lectureOS lives on your Drive
    PROJECT_ROOT = '/content/drive/MyDrive/lectureOS'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

DATA_PATH   = os.path.join(PROJECT_ROOT, 'data', 'train', 'synthetic_qa.jsonl')
ADAPTER_DIR = os.path.join(PROJECT_ROOT, 'models', 'lectureOS_sft_adapter')

os.makedirs(ADAPTER_DIR, exist_ok=True)
print(f'Project root : {PROJECT_ROOT}')
print(f'Training data: {DATA_PATH}')
print(f'Adapter out  : {ADAPTER_DIR}')

## 1 · Install dependencies

In [ ]:
%pip install -q \
    transformers==4.47.1 \
    peft==0.14.0 \
    bitsandbytes==0.45.0 \
    accelerate==1.2.1 \
    trl==0.13.0 \
    datasets==3.2.0 \
    matplotlib

## 2 · Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Switch to a GPU runtime: Runtime → Change runtime type → T4/A100.')

print(f'CUDA device : {torch.cuda.get_device_name(0)}')
print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3 · Load & format the training data

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    'You are lectureOS, an AI teaching assistant. '
    'Answer student questions accurately and concisely based on lecture material.'
)

def load_jsonl(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def format_chat(example: dict) -> dict:
    """Convert QA pair to Qwen2.5 chat-template string."""
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': example['question']},
        {'role': 'assistant', 'content': example['answer']},
    ]
    return {'messages': messages}

raw = load_jsonl(DATA_PATH)
print(f'Loaded {len(raw):,} examples')
print('Sample:', json.dumps(raw[0], indent=2))

In [ ]:
records = [format_chat(ex) for ex in raw]
dataset = Dataset.from_list(records)

split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds  = split['test']

print(f'Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}')

## 4 · Load model in 4-bit NF4

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,   # QLoRA double-quant
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False          # required for gradient checkpointing
model.config.pretraining_tp = 1

print('Model loaded.')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## 5 · Apply LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6 · Configure SFTTrainer & train

In [ ]:
from trl import SFTTrainer, SFTConfig

# SFTTrainer expects a column of already-templated strings; we apply the
# tokenizer's chat template on the fly via formatting_func.
def formatting_func(examples: dict) -> list[str]:
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return texts

sft_config = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch = 16
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_seq_length=1024,
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',                   # set to 'wandb' if desired
    dataset_text_field=None,            # we use formatting_func
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=formatting_func,
    tokenizer=tokenizer,
)

print('Starting training…')
train_result = trainer.train()

## 7 · Save adapter

In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to: {ADAPTER_DIR}')
print('Files:', os.listdir(ADAPTER_DIR))

## 8 · Plot training loss

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps  = [e['step'] for e in log_history if 'loss' in e]
train_losses = [e['loss'] for e in log_history if 'loss' in e]
eval_steps   = [e['step'] for e in log_history if 'eval_loss' in e]
eval_losses  = [e['eval_loss'] for e in log_history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_steps, train_losses, label='Train loss', color='steelblue')
if eval_losses:
    ax.plot(eval_steps, eval_losses, label='Eval loss',  color='tomato', marker='o')
ax.set_xlabel('Step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('lectureOS QLoRA — SFT Loss Curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ADAPTER_DIR, 'loss_curve.png'), dpi=150)
plt.show()
print(f'Final train loss : {train_losses[-1]:.4f}')
if eval_losses:
    print(f'Final eval loss  : {eval_losses[-1]:.4f}')

## 9 · Quick inference test

Reload the adapter on top of the quantised base model and ask one question.

In [ ]:
from peft import PeftModel
from transformers import pipeline

# Merge adapter back for inference (stays 4-bit)
inf_model = PeftModel.from_pretrained(model, ADAPTER_DIR)
inf_model.eval()

TEST_QUESTION = 'What is the role of rank r in LoRA?'

messages = [
    {'role': 'system',  'content': SYSTEM_PROMPT},
    {'role': 'user',    'content': TEST_QUESTION},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = inf_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

# Strip the prompt tokens from the response
response_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
response = tokenizer.decode(response_ids, skip_special_tokens=True)

print('=' * 60)
print(f'Q: {TEST_QUESTION}')
print('-' * 60)
print(f'A: {response}')
print('=' * 60)

## 10 · Training summary

In [ ]:
metrics = train_result.metrics
trainer.log_metrics('train', metrics)
trainer.save_metrics('train', metrics)

print('\n--- Run summary ---')
print(f"  Total steps       : {metrics.get('train_steps_per_second', '?')}")
print(f"  Training loss     : {metrics.get('train_loss', '?'):.4f}")
print(f"  Runtime           : {metrics.get('train_runtime', 0)/60:.1f} min")
print(f"  Adapter location  : {ADAPTER_DIR}")